## The Languages of the World: An Analysis of Glottolog Data

By Yappers (Angelo, Jinglei, Nandini, Rhea, Rauf, Parthavi)

## 1. Introduction

This report explores and analyses data from the TidyTuesday Project’s “The Languages of the World” dataset, curated from Glottolog. With information on over 8,000 languages, this exploration aims to uncover global patterns of language endangerment and extinction. The dataset includes language names, broad geographic regions, location data, family groupings and endangerment status, allowing both linguistic and spatial patterns to be examined. Using this dataset, the report aims to identify global patterns in language endangerment and examine how language relationships and geographic distribution influence vulnerability.

Current linguistic research suggests that language loss is not random; rather, it is driven by specific geographical and genealogical factors. Studies by Amano et al. (2014) and Bromham (2023) have utilised models from conservation biology to show that small "population sizes" (speaker numbers) and restricted geographical ranges are the primary predictors of extinction risk. Furthermore, Urban (2021) notes that language isolates often cluster in specific geographical hotspots like the Americas, where they may be uniquely vulnerable to shifting socio-economic landscapes.

Building on these scholarly frameworks, this report investigates- **How do language isolation, family structure, and geographic location interact to influence language endangerment?** It does so with the following questions:

1. Is Isolation a Risk Factor? We examine if being a genealogical isolate (having no known relatives) inherently increases a language's vulnerability.

2. Geographical Patterns: What is the Geographical Distribution of Isolated Languages by Endangerment Status? This tests the "hotspot" theories proposed by Urban (2021).

3. Regional Context (North America): How does the level of language endangerment vary with language family size in North America?This is identified as a region identified by Amano et al. (2014) as facing intense pressure.


### The Data

The dataset merged information from several CSV files from Glottolog, including `Endangered_status.csv`, `families.csv`, and `languages.csv`.

The relevant variables of interest have been listed below:

| Variable | Type | Description |
| :--- | :--- | :--- |
| id | Character | Unique identifier for the language |
| name | Character | Name of the language |
| macroarea | Character | General geographic area where the language is located |
| latitude | Double | Latitude of the language's location |
| longitude | Double | Longitude of the language's location |
| is_isolate | Boolean | Whether the language is a language isolate |
| status_code | Integer (1-6) | Agglomerated Endangerment Status code |
| status_label | Categorical | Description of the endangerment category |

<br></br>
***Agglomerated Endangerment Status: Codes and Description***

| Code | Description |
| :--- | :--- |
| 1 | Not Endangered |
| 2 | Threatened |
| 3 | Shifting |
| 4 | Moribund |
| 5 | Nearly Extinct |
| 6 | Extinct |


In [ ]:
import pandas as pd

# Load and clean endangered_status
# Reading from the raw URL
raw_values_url = "https://raw.githubusercontent.com/glottolog/glottolog-cldf/refs/heads/master/cldf/values.csv"
endangered_status = pd.read_csv(raw_values_url)

# Filtering for Parameter_ID == "aes" (Agglomerated Endangerment Status)
endangered_status = endangered_status[endangered_status['Parameter_ID'] == 'aes']

# Selecting and Renaming
endangered_status = endangered_status[['Language_ID', 'Value', 'Code_ID']].rename(
    columns={'Language_ID': 'id', 'Value': 'status_code', 'Code_ID': 'status_label'}
)

# String cleaning (Removing "aes-" prefix and replacing "_" with space)
endangered_status['status_label'] = (
    endangered_status['status_label']
    .str.replace(r'^aes-', '', regex=True)
    .str.replace('_', ' ')
)

In [ ]:
# Load Language and Family data
raw_lgs_url = "https://raw.githubusercontent.com/glottolog/glottolog-cldf/refs/heads/master/cldf/languages.csv"
fam_lgs = pd.read_csv(raw_lgs_url)

# Filter and clean Families
families = fam_lgs[fam_lgs['Level'] == 'family'].copy()
families = families[['ID', 'Name']].rename(columns={'ID': 'id', 'Name': 'family'})
# Ensure all column names are lowercase (dplyr's rename_with equivalent)
families.columns = families.columns.str.lower()

In [ ]:
# Filter and clean Languages
languages = fam_lgs[fam_lgs['Level'] == 'language'].copy()
cols_to_keep = ['ID', 'Name', 'Macroarea', 'Latitude', 'Longitude',
                'ISO639P3code', 'Countries', 'Is_Isolate', 'Family_ID']
languages = languages[cols_to_keep]

# Lowercase all column names
languages.columns = languages.columns.str.lower()

In [ ]:
#df_families = languages[languages['family_id'].notna()].copy() # filter for rows where languages is NOT NaN

df_families = languages.copy()
df_families['is_isolate'] = df_families['family_id'].isna()
# Merging cleaned Languages and Endangered_Status data
table1 = df_families.merge(endangered_status, on = 'id', how = 'left')
table1 = df_families.merge(endangered_status, on = 'id', how = 'left')

In [ ]:
# Merging new data with Families data
table2 = table1.merge(families, left_on = 'family_id', right_on = 'id', how = 'left')

In [ ]:
yappers = table2.drop(columns='id_y').rename(columns={'id_x': 'id'})
print(yappers.head())
print(f"Please use feel free to use yappers as your cleaned data. it contains all the relevant columns we need :) Thanks!")

## 2. Data Cleaning and Summary

Our analysis leverages data from Glottolog, a comprehensive online catalog of the world's languages. The initial dataset was constructed by merging information from several Glottolog CSV files, specifically `values.csv` and `languages.csv`. We began by loading `values.csv` to extract language endangerment statuses, focusing on the 'Agglomerated Endangerment Status' (`Parameter_ID == 'aes'`). This involved filtering the data, renaming columns to `id`, `status_code`, and `status_label`, and cleaning `status_label` by removing prefixes and replacing underscores to ensure consistency.

Concurrently, `languages.csv` was loaded to obtain language-specific details such as names, macroarea, geographical coordinates, ISO codes and family IDs. This file was further processed to differentiate between language entries and family entries, resulting in separate `languages` and `families` dataframes. A crucial cleaning step involved creating an `is_isolate` column within the `languages` dataframe, indicating whether a language truly stands alone (`True` if `family_id` is null) or belongs to a larger family (`False`). Finally, these cleaned dataframes were merged: `languages` with `endangered_status` (on `id`) and then with `families` (on `family_id`) to consolidate all relevant information into our primary `yappers` dataframe.

After these cleaning and merging steps, we removed any duplicate language entries to ensure each unique language was represented only once. This process resulted in a final `yappers` dataframe containing `8618` unique language entries and `12` columns, down from a larger initial pool. The resulting dataframe consolidates language-level data enriched with endangerment status and family information, making it ready for our analysis. From a preliminary overview of the `yappers` dataframe, we noted key characteristics, such as the distribution of `is_isolate` values, indicating the count of isolated languages versus those in families, and the counts across various `status_label` categories, which provide an initial glimpse into the prevalence of different endangerment levels. Moreover, within the 'status_label' column, we decided to ignore the NaN values, as it is only contributes to about 4% of the dataset.

In [ ]:
print("Shape of the final dataframe (yappers):")
print(yappers.shape)
print("\nDescriptive statistics for numerical columns:")
display(yappers.describe())
print("\nValue counts for 'status_label':")
display(yappers['status_label'].value_counts(dropna=False))
print("\nValue counts for 'is_isolate':")
display(yappers['is_isolate'].value_counts(dropna=False))

## 3. Visualisations

### Plot 1: Is Isolation a Risk Factor?

This stacked bar chart compares the endangerment status distribution between isolated languages and non-isolated languages. It directly explores whether being an 'isolate' increases a language's risk of endangerment.

*   **Variables Used**: The primary categorical variable is `is_isolate` (transformed to 'Isolate' or 'Not Isolate'), which forms the main bars. The `status_label` (endangerment levels like 'not endangered', 'threatened', 'extinct') constitutes the stacked segments within each bar.
*   **Why this visualisation?**: A stacked bar chart is ideal for showing the composition of different categories (endangerment statuses) within two main groups (isolated vs. non-isolated) and for comparing their proportional distributions. By normalising the bars to 100%, it allows for a clear visual comparison of the *percentage* of languages in each endangerment category for both isolated and non-isolated groups, making it easy to identify shifts in endangerment profiles.

In [ ]:
#plot 1
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

import plotly.express as px

yappers_plt1 = yappers.copy()

danger_palette = {
    'not endangered': '#2ecc71',
    'threatened': '#f1c40f',
    'shifting': '#e67e22',
    'moribund': '#e74c3c',
    'nearly extinct': '#9b2226',
    'extinct': '#1a1a1a'
}
status_order = ["not endangered", "threatened", "shifting", "moribund", "nearly extinct", "extinct"]

yappers_plt1['is_isolate'] = yappers_plt1['is_isolate'].map({True: 'Isolate', False: 'Not Isolate', 1: 'Isolate', 0: 'Not Isolate'})
df_plot = yappers_plt1.groupby(['is_isolate', 'status_label']).size().reset_index(name='count')
df_totals = df_plot.groupby('is_isolate')['count'].transform('sum')
df_plot['percentage'] = (df_plot['count'] / df_totals * 100).round(1)

fig = px.bar(
    df_plot,
    x="is_isolate",
    y="percentage",
    color="status_label",
    title="Is Isolation a Risk Factor?",
    labels={"percentage": "Percentage of Languages (%)", "is_isolate": "Language Type"},
    category_orders={"status_label": status_order, "is_isolate": ["Isolate", "Not Isolate"]},
    color_discrete_map=danger_palette,
    text=df_plot.apply(lambda r: f"{int(r['count'])}({r['percentage']}%)", axis=1),
    hover_data={"count": True, "percentage": True, "is_isolate": False}
)

fig.update_traces(textposition='inside', textfont_size=12)
fig.update_layout(
    xaxis_title="",
    yaxis_ticksuffix="%",
    legend_title_text='Endangerment Status',
    height=700,
    width=1000
)
plt.savefig("chart1.png", bbox_inches="tight", dpi=300)
fig.show()


### Plot 2: What is the Geographical Distribution of Isolated Languages by Endangerment Status?

Plot 2, a scatter plot, illustrates the geographical distribution of isolated languages classified under non-critical endangerment levels, namely not endangered, threatened and shifting. Each point represents an isolated language, positioned using its longitude and latitude coordinates. The inclusion of the world map background provides geographical context, allowing for clearer interpretation of spatial patterns and regional clustering of these languages.

*   **Variables Used**: `longitude` and `latitude`define the geographical position of each language and are plotted on the x-axis and y-axis respectively.
`status_label` represents the level of endangerment and is used to colour the points, distinguishing between not endangered, threatened, and shifting categories.
The dataset is filtered using `is_isolate == Isolate` to ensure that only language isolates are included in this visualisation.
*   **Why this visualisation?**: A scatter plot is well-suited for mapping individual language isolates across global coordinates, allowing for direct visualisation of spatial distribution. The use of colour encoding for `status_label` enables comparison across different levels of endangerment, while the addition of the world map background enhances interpretability by situating the data within real geographic regions. This makes it easier to identify clustering patterns and potential areas of vulnerability.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

sns.set_theme(style="whitegrid")

critical_status = ['moribund', 'nearly extinct', 'extinct']
non_critical_status = ['not endangered', 'threatened', 'shifting']

plot_map = yappers.dropna(subset=['longitude', 'latitude', 'status_label', 'is_isolate']).copy()

critical_isolates = plot_map[
    (plot_map['is_isolate'] == True) &
    (plot_map['status_label'].isin(critical_status))
]

non_critical_isolates = plot_map[
    (plot_map['is_isolate'] == True) &
    (plot_map['status_label'].isin(non_critical_status))
]

print("Critical isolates:", critical_isolates.shape)
print("Non-critical isolates:", non_critical_isolates.shape)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"
world = gpd.read_file(url)

world.plot(ax=ax[0], color='#e8e8e8', edgecolor='white')
world.plot(ax=ax[1], color='#e8e8e8', edgecolor='white')

danger_palette = {
    'not endangered': '#2ecc71',
    'threatened': '#f1c40f',
    'shifting': '#e67e22',
    'moribund': '#e74c3c',
    'nearly extinct': '#9b2226',
    'extinct': '#1a1a1a'
}

critical_order = ['moribund', 'nearly extinct', 'extinct']
non_critical_order = ['not endangered', 'threatened', 'shifting']

sns.scatterplot(
    data=critical_isolates,
    x="longitude", y="latitude",
    hue="status_label", hue_order=critical_order,
    palette=danger_palette,
    alpha=0.8, s=80, edgecolor='w',
    ax=ax[0], zorder=5
)
ax[0].set_title("Critical Status (Moribund to Extinct)", fontsize=13, fontweight='bold')

sns.scatterplot(
    data=non_critical_isolates,
    x="longitude", y="latitude",
    hue="status_label", hue_order=non_critical_order,
    palette=danger_palette,
    alpha=0.8, s=80, edgecolor='w',
    ax=ax[1], zorder=5
)
ax[1].set_title("Non-Critical Status (Not Endangered to Shifting)", fontsize=13, fontweight='bold')

for a in ax:
    a.set_xlabel("Longitude")
    a.set_ylabel("Latitude")
    a.set_xlim(-180, 180)
    a.set_ylim(-90, 90)

ax[0].legend(title="Status", bbox_to_anchor=(1.05, 1), loc='upper left')
ax[1].legend(title="Status", bbox_to_anchor=(1.05, 1), loc='upper left')

fig.suptitle("Geographical Distribution of Language Isolates by Endangerment Status", fontsize=16, fontweight='bold', y=0.8)
plt.tight_layout()
plt.savefig("chart2.png", bbox_inches="tight", dpi=300)
plt.show()

### Plot 3: How does the level of language endangerment vary with language family size in North America?

Plot 3, a violin plot, visualises how language endangerment varies across different language family sizes (small, medium, large) within North America. It allows us to compare the distribution and severity of endangerment across these categories and assess whether languages from smaller families are more vulnerable to higher levels of endangerment.

*   **Variables Used**: `count`: This variable represents the number of languages within each language family.  Since isolated languages do not belong to any family (i.e. they have missing family_id values), their corresponding count values were explicitly assigned a value of 1. `family_size`: Languages were grouped into three categories based on family size: small (<10 languages), medium (10–99 languages), and large (≥100 languages). These groups contain 806, 2001 and 5311 languages respectively. The dataset is filtered using macroarea == 'North America' to focus the analysis on a specific region.

*   **Why this visualisation**:
A violin plot is well-suited for comparing distributions across categories, as it shows both the density and spread of the data. It captures the full variation in endangerment levels within each family size group. The inclusion of inner boxplots highlights medians and quartiles, providing additional statistical context.



In [ ]:
plot_3 = yappers.copy()

family_size = (
    plot_3.groupby('family_id')
    .agg(count=('family_id', 'size'))
    .reset_index()
)
family_size = family_size.sort_values(by='count').reset_index(drop=True)
plot_3 = plot_3.merge(family_size, on='family_id', how='left')
plot_3['count'] = plot_3['count'].fillna(1).astype(int)
plot_3.shape

In [ ]:
plot_3['family_size'] = 'large'
plot_3.loc[plot_3['count'] < 100, 'family_size'] = 'medium'
plot_3.loc[plot_3['count'] < 10, 'family_size'] = 'small'

large = plot_3[plot_3['family_size'] == 'large']  #5311 families
medium = plot_3[plot_3['family_size'] == 'medium']  #2001 families
small = plot_3[plot_3['family_size'] == 'small']  #806 families
print(large.shape, medium.shape, small.shape)

In [ ]:
status_order = ["not endangered", "threatened", "shifting", "moribund", "nearly extinct", "extinct"]

plot_3['status_label'] = pd.Categorical(
    plot_3['status_label'],
    categories=status_order,
    ordered=True
)

mapping = {label: i+1 for i, label in enumerate(status_order)}
plot_3['status_code'] = plot_3['status_label'].map(mapping)

In [ ]:
north_america = plot_3[plot_3['macroarea'] == 'North America']
order = ['small', 'medium', 'large']

plt.figure(figsize=(8, 5))

sns.violinplot(
    data=north_america,
    x='family_size',
    y='status_label',
    order=order,
    inner='box')

sns.stripplot(
    data=north_america,
    x='family_size',
    y='status_label',
    order=order,
    color='black',
    size=3,
    alpha=0.4
)
plt.gca().invert_yaxis()

plt.xlabel("Family Size", fontsize=12)
plt.ylabel("Endangerment Status", fontsize=12)
plt.title("Endangerment Distribution by Family Size in North America", fontsize=13)


plt.tight_layout()
plt.savefig("chart3.png", bbox_inches="tight", dpi=300)
plt.show()


### Overall Discussion and Conclusion

### Insights of Plot 1: Exploring the relationship between the isolated state of a language and its endangered status

The visualisation indicates the linguistic isolation is strongly associated with a higher risk of language extinction. 43.7% of isolate languages are extinct but non-isolate languages only got 14.4%. By. presenting normalised percentages, the stacked bar chart enables a clear comparison between the two languages groups, highlighting isolation as a significant risk factor. This observe pattern can be explained by demographic, sociopolitical and historical factos. Language isolates are spoken by small and geographically remote communities. Unlike large families language group, they lack linguistic relatives that facilitate cultural continuity, shared resources or revitalization initiatives (Campbell & Rehg, 2018). External pressures such as colonization, globalization and the dominance of major world languages further accelerate language shift and decline (Austin & Sallabank, 2015). Therefore, the data supports the conclusion that linguistic isolation increases vulnerability to extinction.

### Insights of Plot 2: Geographical Distribution of Isolated Languages by Endangerment Status

Plot 2 shows that critically endangered language isolates are not evenly distributed, but cluster in specific regions. Higher concentrations appear in North America, South America and parts of Australia, suggesting the presence of geographical “hotspots” of vulnerability.
South America shows clustering, particularly in more remote areas, where increasing external influence and language shift may be contributing to decline. In contrast, regions like Papunesia and parts of Africa show more non-critical isolates, suggesting relatively stronger language continuity or less intense assimilation pressures.
In North America, the dense presence of critically endangered isolates reflects well-documented historical pressures such as colonisation, language suppression and the dominance of English. This pattern motivates the deeper regional focus in Plot 3.
Overall, the spatial distribution suggests that geographical and historical context plays a crucial role in language endangerment, rather than isolation alone. The clustering of critically endangered isolates in specific regions indicates that external socio-political pressures amplify the risks associated with linguistic isolation.

### Insights of Plot 3: Zooming into North America: Endangerment and Family Size

Plot 3, “Genealogical Context and Language Vitality,” provides a structural perspective on how family size relates to language endangerment in North America. By comparing small, medium and large language families, clear patterns emerge: small families (including isolates) are concentrated among extinct and nearly extinct languages, while medium families show mixed trajectories, with both extinct and shifting/threatened languages. In contrast, large families are mostly clustered around shifting and threatened, with fewer extinct languages, suggesting they may be somewhat more resilient. Overall, the plot shows that languages from smaller families are at the greatest risk, pointing to where preservation efforts may be most needed.

### Summary

Based on the findings from the three plots, the study concludes that isolation and small family size are possible risk factors for language endangerment. The data reveals that language isolates are significantly more prone to extinction, likely because they lack the "safety net" of linguistic relatives to facilitate revitalisation. This vulnerability is heavily influenced by geographical context, as endangered isolates cluster in "hotspots" like the Americas, which are regions where the historical pressures of colonialism and globalisation are most intense. Conversely, isolates in remote, geographically sheltered areas show higher stability. Our study of North America gives us a possible insight of a small family size being a possible risk factor, however we acknowledge that this may not be true for other regions. Ultimately, North America's analysis confirms a clear correlation between scale and survival: smaller language families face a disproportionately higher risk, suggesting that linguistic diversity is most fragile when it lacks both genealogical depth and geographic isolation from globalising forces.

## Teamwork (Task Allocation)

The work for this project was distributed evenly across all group members.

*   **Data Cleaning and Plot 2**: Angelo handled the data cleaning and preprocessing, as well as co-developing the second visualisation - the geographical scatter plot of isolated languages - together with Rauf.
*   **Plot 1**: Jinglei and Nandini collaborated on the first visualisation, the stacked bar chart comparing endangerment status between isolated and non-isolated languages.
*   **Plot 3**: Rhea and Parthavi worked together on the third visualisation, the violin plot examining language endangerment across family sizes in North America.

The discussion and insights for all three visualisations were a collective effort by all six members.

## Reference List

Amano, T., Sandel, B., Eager, H., Bulteau, E., Svenning, J.-C., Dalsgaard, B., Rahbek, C., Davies, R. G., & Sutherland, W. J. (2014). Global distribution and drivers of language extinction risk. Proceedings of the Royal Society B: Biological Sciences, 281(1793), 20141574. https://doi.org/10.1098/rspb.2014.1574

Austin, P. K., & Sallabank, J. (Eds.). (2015). The Cambridge handbook of endangered languages. Cambridge University Press.
https://doi.org/10.1017/CBO9780511975981

Bromham, L. (2023). Language endangerment: using analytical methods from conservation biology to illuminate loss of linguistic diversity. Cambridge Prisms: Extinction, 8(4), 1–30. https://doi.org/10.1017/ext.2022.3

Campbell, L., & Rehg, K. L. (Eds.). (2018). The Oxford handbook of endangered languages. Oxford University Press.
https://doi.org/10.1093/oxfordhb/9780190610029.001.0001

Jonthegeek. (2025). The Languages of the World. GitHub.
https://github.com/rfordatascience/tidytuesday/blob/main/data/2025/2025-12-23/readme.md

Urban, M. (2021). The geography and development of language isolates. Royal Society Open Science,
8(4). https://doi.org/10.1098/rsos.202232
